In [0]:
import pyspark.sql
from pyspark.sql.functions import concat, col, lit, avg
from pyspark.sql.window import Window
from pyspark.sql.types import DecimalType, FloatType

In [0]:
spark_df = spark.read.table("workspace.silver.silver_stocks_prices")

average_close = Window.partitionBy("Ticker").orderBy("Date").rowsBetween(-6, 0)

spark_df = spark_df.withColumn("daily_return", (col('Close') - col('Open')) / col('Open') * 100)
spark_df = spark_df.withColumn("average_close", avg(col("Close")).over(average_close))
spark_df = spark_df.withColumn("price_range", col("High") - col("Low"))

spark_df.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.gold_stocks_prices")